# S-PDO-adjacents — viewer

Two pre-registered questions about the neighbourhood of the live PDO sleeve:

- **(a)** regime threshold **−10 %** (live) vs **−7 %** (predecessor) → **KEEP −10 %**
- **(b)** the **CDO retouch** variant → **KILL**

Pre-registration: `README.md`. Verdicts and honest discussion: `findings.md`.
This notebook only reads `results/` — it never opens prod.db.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

R = Path.cwd() / 'results'
if not R.exists():
    R = Path(__file__).resolve().parent / 'results' if '__file__' in dir() else R
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)
print(sorted(p.name for p in R.glob('*')))

## 0. Parity gate
Study features vs the live sleeve's own functions under a frozen clock, **exact float equality**. This must pass or nothing below is reportable.

In [ ]:
parity = json.loads((R / 'parity.json').read_text())
print('passed:', parity['passed'], ' total checks:', parity['total_checks'])
display(pd.DataFrame(parity['assets']).T)

## (a) Regime threshold — clause table

In [ ]:
qa = json.loads((R / 'qa_clauses.json').read_text())
cl = qa['clauses']
rows = [
    ('A1 BTC OOS delta bp', cl['A1_oos_delta_btc_bp'], '>= +5.0', cl['A1_fired']),
    ('A1 ETH OOS delta bp', cl['A1_oos_delta_eth_bp'], '>= +5.0', cl['A1_fired']),
    ('A2 BTC IS delta bp',  cl['A2_is_delta_btc_bp'],  '>= 0.0',  cl['A2_fired']),
    ('A2 ETH IS delta bp',  cl['A2_is_delta_eth_bp'],  '>= 0.0',  cl['A2_fired']),
]
display(pd.DataFrame(rows, columns=['clause', 'measured', 'required', 'fired']))
print('VERDICT:', qa['verdict'])
print('discriminating trades  total:', qa['band_trades_total'], ' OOS:', qa['band_trades_oos'])

### Mean per-trade outcome by asset × period × threshold (18 bp round trip)

In [ ]:
s = pd.read_csv(R / 'qa_summary.csv')
piv = s.pivot_table(index=['asset', 'period'], columns='config',
                    values=['n', 'mean_bp_18'])
piv[('delta_bp', '')] = piv[('mean_bp_18', 'm7')] - piv[('mean_bp_18', 'm10')]
display(piv.round(2))

### Why the OOS clause measured exactly zero
Every out-of-sample entry sat far above both thresholds, and the OOS window spent almost no time in the discriminating [−10 %, −7 %) band.

In [ ]:
display(pd.read_csv(R / 'qa_oos_trade_list.csv')[
    ['asset', 'config', 'entry_iso', 'gap_pct', 'btc_30d_pct', 'net_bp', 'in_band']])
display(pd.read_csv(R / 'qa_regime_occupancy.csv').round(4))

### The 15 discriminating trades (taken at −10 %, blocked at −7 %)

In [ ]:
band = pd.read_csv(R / 'qa_band_trade_list.csv')
display(band[['asset', 'entry_iso', 'gap_pct', 'btc_30d_pct', 'net_bp']])
display(pd.read_csv(R / 'qa_band_trades.csv').round(2))

### Substitution: −7 % is not a subset of −10 %
Blocking a setup frees the one-trade-per-day slot, so a later touch bar the same day can fire instead — usually at a worse price.

In [ ]:
t = pd.read_csv(R / 'qa_trades.csv')
for a in ['BTC', 'ETH']:
    s10 = set(t[(t.asset == a) & (t.config == 'm10')].entry_ts)
    s7 = set(t[(t.asset == a) & (t.config == 'm7')].entry_ts)
    only7 = t[(t.asset == a) & (t.config == 'm7') & (t.entry_ts.isin(s7 - s10))]
    print(f'== {a}: removed {len(s10 - s7)}, added {len(s7 - s10)}')
    if len(only7):
        display(only7[['entry_iso', 'btc_30d_pct', 'net_bp']])

### Equity curves, both thresholds

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, a in zip(axes, ['BTC', 'ETH']):
    for cfg, lbl in [('m10', 'threshold -10%'), ('m7', 'threshold -7%')]:
        d = t[(t.asset == a) & (t.config == cfg)].sort_values('entry_ts')
        ax.plot(pd.to_datetime(d.entry_iso), d.net_bp.cumsum(), label=lbl)
    ax.axhline(0, lw=0.8, color='0.6')
    ax.set_title(f'{a} PDO cumulative net bp (18bp RT)')
    ax.legend()
fig.tight_layout()

## (b) CDO retouch — clause table

In [ ]:
qb = json.loads((R / 'qb_clauses.json').read_text())
c = qb['clauses']
rows = [
    ('B1  bootstrap p05 of mean R', c['B1_mean_R_p05'], '> 0', c['B1_fired']),
    ('B2  positive complete folds', f"{c['B2_positive_folds']} of {c['B2_complete_folds']}", '>= 3 of 4', c['B2_fired']),
    ('B3  DSR @ N_TRIALS=2', c['B3_dsr'], '> 0.95', c['B3_fired']),
]
display(pd.DataFrame(rows, columns=['clause', 'measured', 'required', 'fired']))
print('VERDICT:', qb['verdict'])
print('n trades:', qb['n_trades'], ' mean R:', round(qb['mean_R'], 4),
      ' sum R:', round(qb['sum_R'], 1), ' maxDD R:', round(qb['max_drawdown_R'], 1))
print(json.dumps(qb['robustness_bounds'], indent=1))

### Walk-forward folds

In [ ]:
display(pd.read_csv(R / 'qb_folds.csv').round(4))

### Every slice is negative

In [ ]:
qs = pd.read_csv(R / 'qb_summary.csv')
display(qs.round(4))

### The mechanism: target hit rate below break-even
2R target, 1R stop, 18 bp = 0.18 R cost → break-even target rate 39.3 %.

In [ ]:
tb = pd.read_csv(R / 'qb_trades.csv')
mix = tb.exit_type.value_counts(normalize=True).rename('share')
display(mix.to_frame().round(4))
be = qb['robustness_bounds']['breakeven_target_hit_rate_at_18bp']
print(f"target hit rate {qb['robustness_bounds']['target_hit_rate']:.4f} "
      f"vs break-even {be:.4f}")
print(f"mean R at zero cost: {qb['robustness_bounds']['mean_R_at_zero_cost']:+.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for a in ['BTC', 'ETH']:
    d = tb[tb.asset == a].sort_values('entry_ts')
    ax.plot(pd.to_datetime(d.entry_iso), d.R.cumsum(), label=a)
d = tb.sort_values('entry_ts')
ax.plot(pd.to_datetime(d.entry_iso), d.R.cumsum(), label='pooled', lw=2, color='k')
ax.axhline(0, lw=0.8, color='0.6')
ax.set_title('CDO retouch cumulative R (18bp RT, 1% = 1R)')
ax.legend()
fig.tight_layout()

---
**Verdicts**

- (a) `KEEP -10%` — A1 measured +0.00 bp (no discriminating OOS trades at all), A2 measured −2.01 bp (BTC) and −4.00 bp (ETH), both below their bars. No change to `config.py` is recommended.
- (b) `KILL` — 0 of 3 clauses fired; mean −0.164 R over 1,855 trades, 0 of 4 folds positive, DSR 7.2e-09. At zero cost the mean is +0.016 R, i.e. the signal is a coin flip and the cost is what makes it a loser.

See `findings.md` for the full clause tables, the substitution finding, and the limitations that bound both answers.